# Fixing GAN Mode Collapse with Spectral Normalization for E-commerce Product Image Generation

This notebook runs the whole project on **Google Colab (free T4 GPU)**:

1. Download and prepare the Myntra Fashion Product Images dataset (64x64, top 20 categories)
2. Train the **baseline** DCGAN (BatchNorm discriminator + BCE loss)
3. Train the **improved** GAN (spectral-normalized discriminator + hinge loss)
4. Train the category classifier used for the mode-coverage metric
5. Benchmark both models: FID, precision/recall, categories covered, inference latency
6. Launch the Gradio demo (improved model only)

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

Everything is saved to Google Drive, so if Colab disconnects, re-run the cells. Training resumes from the last checkpoint.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Where everything is kept between sessions
DRIVE_DIR = '/content/drive/MyDrive/gan-mode-collapse'
!mkdir -p "$DRIVE_DIR"/data "$DRIVE_DIR"/runs "$DRIVE_DIR"/checkpoints

In [ ]:
# Replace with your GitHub repo URL
REPO_URL = 'https://github.com/git-shreya/gan-mode-collapse.git'
%cd /content
!rm -rf gan-mode-collapse && git clone -q $REPO_URL
%cd /content/gan-mode-collapse
!pip install -q torch-fidelity datasets gradio
# keep data, runs and checkpoints on Drive
!rm -rf data runs && ln -s "$DRIVE_DIR"/data data && ln -s "$DRIVE_DIR"/runs runs
!ls

## 1. Prepare the dataset (about 5 minutes, only needed once)

In [ ]:
import os
if not os.path.exists('data/fashion64.npz'):
    !python -m src.data --out data/fashion64.npz --top_k 20
else:
    print('Dataset already prepared')
!cat data/fashion64_meta.json

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load('data/fashion64.npz')
names = d['class_names']
fig, axes = plt.subplots(2, 10, figsize=(16, 3.6))
for c, ax in enumerate(axes.flat):
    i = np.where(d['labels'] == c)[0][0]
    ax.imshow(d['images'][i]); ax.set_title(names[c], fontsize=8); ax.axis('off')
plt.suptitle('One real example per category'); plt.show()

## 2. Train the baseline DCGAN

Same settings for both models: 40 epochs, batch 128, Adam (lr 2e-4, betas 0.5/0.999), seed 42.
Expect roughly 25-40 s per epoch on a T4. If Colab disconnects, just run the cell again.

In [ ]:
!python -m src.train --model baseline --data data/fashion64.npz --out runs/baseline --epochs 40 --resume

## 3. Train the improved GAN (spectral normalization + hinge loss)

In [ ]:
!python -m src.train --model improved --data data/fashion64.npz --out runs/improved --epochs 40 --resume

In [ ]:
from IPython.display import Image, display
print('Baseline, final epoch'); display(Image('runs/baseline/samples/epoch_040.png'))
print('Improved, final epoch'); display(Image('runs/improved/samples/epoch_040.png'))

## 4. Train the category classifier (used only for evaluation and the demo filter)

In [ ]:
import os
CLF_DRIVE = f'{DRIVE_DIR}/checkpoints/classifier.pt'
if not os.path.exists(CLF_DRIVE):
    !python -m src.classifier --data data/fashion64.npz --out "$CLF_DRIVE" --epochs 6
!cp "$CLF_DRIVE" checkpoints/classifier.pt

## 5. Benchmark both models

In [ ]:
!python -m src.evaluate --data data/fashion64.npz \
    --baseline runs/baseline/G_final.pt --improved runs/improved/G_final.pt \
    --classifier checkpoints/classifier.pt --n 10000 --out results

In [ ]:
import pandas as pd
display(pd.read_csv('results/metrics.csv', index_col=0).T)
for f in ['side_by_side.png', 'category_distribution.png', 'training_curves.png', 'nearest_neighbours.png']:
    print(f); display(Image('results/' + f))

## 6. Copy the outputs into the repo for GitHub

This copies the final generator, classifier, results and sample grids into the repo folder and makes a zip you can download and push.

In [ ]:
!cp runs/improved/G_final.pt checkpoints/G_improved.pt
!cp runs/baseline/G_final.pt checkpoints/G_baseline.pt
!mkdir -p results/training_logs results/samples
!cp runs/baseline/log.csv results/training_logs/baseline_log.csv
!cp runs/improved/log.csv results/training_logs/improved_log.csv
!cp data/fashion64_meta.json results/dataset_meta.json
!cp runs/baseline/samples/epoch_040.png results/samples/baseline_epoch40.png
!cp runs/improved/samples/epoch_040.png results/samples/improved_epoch40.png
!cp checkpoints/*.pt "$DRIVE_DIR"/checkpoints/
!cd /content && zip -qr gan-mode-collapse-outputs.zip gan-mode-collapse/checkpoints gan-mode-collapse/results
from google.colab import files
files.download('/content/gan-mode-collapse-outputs.zip')

## 7. Launch the demo (improved model only)
Click the public `gradio.live` link that appears below.

In [ ]:
!python app.py --generator checkpoints/G_improved.pt --classifier checkpoints/classifier.pt --share